# Solution E1 · Comparing two forests with a single seed and 25 trees

**Type**: Find the bug

## Objective

Take apart a conclusion based on a single run: a colleague compared `max_features=4` and `max_features=6` with 25 trees and a single `random_state`, and declared "no difference" (or, with another seed, "6 is much better"). Measure the **variability due to the seed** by repeating with 30 seeds, and establish
what can really be concluded.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = make_classification(800, n_features=16, n_informative=5, n_redundant=3, class_sep=0.7, flip_y=0.1, random_state=5)
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def acc(mf, seed, n=25):
    return cross_val_score(RandomForestClassifier(n, max_features=mf, random_state=seed, n_jobs=-1), X, y, cv=cv).mean()

## Commented solution

In [ ]:
diff_seed0 = acc(6, 0) - acc(4, 0)
print(f"difference (max_features=6 minus 4) with a single seed and 25 trees: {diff_seed0:+.4f} -> 'there is no difference, let's keep sqrt'")

**Where the bug is.** The comparison is made on **a single run**, but a forest with 25 trees is a stochastic model: the variability across seeds of the same configuration (about 0.7 points) is larger than the difference the colleague wants to measure (about half a point). With seed 0 the
difference is +0.1 points (no difference), with other seeds it can be +2.6 or −0.9: the conclusion depends on the chosen seed.

In [ ]:
acc4 = np.array([acc(4, s) for s in range(30)])
acc6 = np.array([acc(6, s) for s in range(30)])
differences = acc6 - acc4
print("first differences (max_features 6 - 4):", differences[:8].round(3))

In [ ]:
seed_sd = float(acc4.std())
mean_diff, diff_sd = float(differences.mean()), float(differences.std())
standard_error = diff_sd / np.sqrt(30)
wins6 = int((differences > 0).sum())
sign_changes = bool((differences > 0).any() and (differences < 0).any())
print(f"sd across seeds (max_features=4): {seed_sd:.4f} | mean difference {mean_diff:+.4f} +/- {standard_error:.4f} (sd {diff_sd:.4f}) | max_features=6 wins in {wins6}/30 seeds | the sign changes: {sign_changes}")

In [ ]:
seed_sd_300 = float(np.std([acc(4, s, 300) for s in range(10)]))
sd_reduction = seed_sd / seed_sd_300
print(f"sd across seeds at 25 trees: {seed_sd:.4f} | at 300 trees: {seed_sd_300:.4f} (reduction {sd_reduction:.1f}x)")

## Check

In [ ]:
with verify():
    check(abs(diff_seed0) < 0.01, "the colleague sees a tiny difference (less than one point) with seed 0", "don't modify the colleague's code")
    check(acc4 is not None and len(acc4) == 30 and len(acc6) == 30 and close(differences, np.asarray(acc6) - np.asarray(acc4)), "30 accuracies for each configuration and their differences", "acc(4, s), acc(6, s) for s = 0..29")
    check(0.004 < seed_sd < 0.012 and diff_sd > 0.005, "the standard deviation across seeds is 0.4-1.2 points; that of the differences over half a point", "np.std")
    check(sign_changes is True and wins6 >= 18 and 0.002 < mean_diff < 0.01 and close(standard_error, diff_sd / np.sqrt(30)), "the difference changes sign across seeds; max_features=6 wins in at least 18 seeds out of 30; mean between 0.2 and 1 point", "mean and standard error of the differences")
    check(mean_diff > 2 * standard_error, "the mean difference exceeds two standard errors: it is probably real, but small", "mean_diff / standard_error")
    check(seed_sd_300 is not None and sd_reduction > 1.6, "with 300 trees the variability across seeds drops by over 1.6 times", "sd of acc(4, s, n=300)")
done()

## What you learned

- **A forest is stochastic:** with 25 trees, the same configuration with different seeds varies by about 0.7 points (standard deviation) and the difference between two configurations varies by about 0.9 points from one seed to another. The colleague saw +0.1 with seed 0; with other seeds they would have seen +2.6 or −0.9.
- **The true difference exists but is small:** over 30 seeds `max_features=6` wins 23 times and the mean difference is half a point (3 standard errors). The honest conclusion is "6 is probably a bit better, by about half a point": neither "no difference" nor "much better".
- **With more trees the variability drops** (0.3 points at 300 trees): the seed noise shrinks roughly like `1/√n`, so to compare configurations **you need a high number of trees** (and/or many seeds). With 25 trees the noise of the comparison is as large as the effect you are looking for.
- **Protocol:** fix the same seed for the compared models (**paired** comparison), use 200+ trees, repeat with at least 10 seeds or with a repeated cross-validation, and report the difference with its standard error. And write `random_state` in the code: without it you cannot reproduce anything.

## Common pitfall

Picking the "winner" between two configurations after a single run, maybe with different `random_state` values. With stochastic models it is a random choice, and in a tuning with dozens of configurations you systematically pick the lucky one.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You compared two random forests and one does better by 0.5%. Do you trust it?</summary>

First I measure the variability due to the seed: I repeat with many seeds (same seed for both, paired comparison) and with a high number of trees, and I report the mean difference with the standard error. If the difference is smaller than or of the same order as the noise, I do not consider it a result; if it is distinguishable but small, I
weigh it against the cost (time, complexity).

</details>